# 03 · 🆕 The Future of Parallel Python: Free-Threading (No-GIL) & Subinterpreters

**Module 13 – Concurrency** · Estimated time: 30 min

> ❗ Brand-new material (Python 3.13–3.15), in neither source.

### 🎯 What you will learn
- What "free-threaded" Python is and how to check whether you're using it
- How to install/run it, and what to watch out for
- Subinterpreters with `concurrent.interpreters` and `InterpreterPoolExecutor` (3.14)
- A decision guide for CPU-bound parallelism in 2026

## 1. Free-threaded CPython (PEP 703)

| Version | Status |
|---|---|
| 3.13 (2024) | experimental build without the GIL (`python3.13t`) |
| 3.14 (2025) | **officially supported** (PEP 779), still an optional, separate build |
| 3.15 (2026) | the official **macOS installer installs free-threading support by default**; a stable ABI for free-threaded extension modules (PEP 803/820/793) makes it easier for libraries to support it |

In a free-threaded build, **threads can run Python code on multiple cores at the same time**, so CPU-bound threading code finally speeds up.

### Am I running a free-threaded build?

In [1]:
import sys, sysconfig

free_threaded_build = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
gil_enabled = sys._is_gil_enabled() if hasattr(sys, "_is_gil_enabled") else True
print("Python:", sys.version.split()[0])
print("Free-threaded build:", free_threaded_build)
print("GIL currently enabled:", gil_enabled)

Python: 3.15.0
Free-threaded build: False
GIL currently enabled: True


### How to get it
- **python.org installers**: on Windows tick *"Download free-threaded binaries"*; on macOS 3.15 it's installed by default as `python3.15t`
- **uv**: `uv python install 3.15t` then `uv run --python 3.15t script.py`
- **Linux distros / conda-forge**: look for `python3.15t` / `python-freethreading` packages
- You can force the GIL back on with `PYTHON_GIL=1` or `-X gil=1`

### A CPU-bound benchmark that shows the difference

Run this file with a regular and a free-threaded interpreter and compare:

In [2]:
%%writefile ft_bench.py
import sys, time
from concurrent.futures import ThreadPoolExecutor

def count_primes(limit):
    return sum(1 for n in range(2, limit) if all(n % d for d in range(2, int(n ** 0.5) + 1)))

if __name__ == "__main__":
    jobs = [200_000] * 4
    t = time.perf_counter()
    [count_primes(j) for j in jobs]
    seq = time.perf_counter() - t

    t = time.perf_counter()
    with ThreadPoolExecutor(4) as ex:
        list(ex.map(count_primes, jobs))
    thr = time.perf_counter() - t

    gil = sys._is_gil_enabled() if hasattr(sys, "_is_gil_enabled") else True
    print(f"GIL enabled={gil}: sequential {seq:.2f}s, 4 threads {thr:.2f}s, speed-up x{seq / thr:.1f}")

Writing ft_bench.py


In [3]:
import subprocess
print(subprocess.run([sys.executable, "ft_bench.py"], capture_output=True, text=True).stdout)

GIL enabled=True: sequential 0.65s, 4 threads 0.61s, speed-up x1.1



With the regular (GIL) build above, 4 threads give **no speed-up**. On a free-threaded build (`python3.15t ft_bench.py`) you'd typically see roughly a **3–4× speed-up** on a 4+ core machine.

### Things to know
- Single-threaded code can be slightly slower on the free-threaded build.
- **C extension modules** must explicitly support free-threading; if an incompatible one is imported, CPython may re-enable the GIL (with a warning). Most popular packages (NumPy, etc.) now ship free-threaded wheels.
- Without the GIL, **you** must protect shared mutable data with locks (Module 13 · 01). Built-in types like `list` and `dict` stay internally consistent, but compound operations (`check-then-update`) still need a `Lock`.

## 2. Subinterpreters: `concurrent.interpreters` (3.14, PEP 734)

A **subinterpreter** is an isolated Python interpreter inside the same process, each with its **own GIL**. That gives true parallelism like multiprocessing, but with lower overhead than separate processes.

In [4]:
from concurrent import interpreters

interp = interpreters.create()
interp.exec("import sys; greeting = 'Hello from a subinterpreter!'")   # run code inside it
print(interp.call(pow, 2, 20))            # call a function inside it and get the result back
interp.close()

1048576


> 💡 In Jupyter, `print()` *inside* a subinterpreter writes to the terminal that launched Jupyter, not to the notebook. Return values with `call()` or a queue (below) instead.

Interpreters don't share objects; data passes through **queues** (copied/shared safely):

In [5]:
interp = interpreters.create()
q = interpreters.create_queue()
interp.prepare_main(q=q)                  # make the queue available inside the interpreter
interp.exec("q.put(sum(range(1_000_000)))")
print("result from subinterpreter:", q.get())
interp.close()

result from subinterpreter: 499999500000


### The easy way: `InterpreterPoolExecutor` (3.14)

Same API as thread/process pools:

In [6]:
%%writefile prime_tasks.py
def count_primes(limit):
    return sum(1 for n in range(2, limit) if all(n % d for d in range(2, int(n ** 0.5) + 1)))

Writing prime_tasks.py


In [7]:
import time
from concurrent.futures import InterpreterPoolExecutor, ThreadPoolExecutor
from prime_tasks import count_primes

jobs = [200_000] * 4
for name, Pool in [("threads", ThreadPoolExecutor), ("interpreters", InterpreterPoolExecutor)]:
    t = time.perf_counter()
    with Pool(max_workers=4) as ex:
        results = list(ex.map(count_primes, jobs))
    print(f"{name:<13} {time.perf_counter() - t:.2f}s  {results[0]}")

threads       0.62s  17984
interpreters  0.19s  17984


## 3. Decision guide (2026)

| Need | Best choice |
|---|---|
| I/O-bound (network, disk) | `asyncio` or `ThreadPoolExecutor` |
| CPU-bound, simple, any Python | `ProcessPoolExecutor` |
| CPU-bound, isolated tasks, lower overhead | `InterpreterPoolExecutor` (3.14+) |
| CPU-bound with shared in-memory data | free-threaded build + threads (+ locks) |
| Numeric/array work | NumPy / vectorised libraries |

## 📝 Summary
- Free-threaded Python removes the GIL: officially supported since 3.14; on macOS 3.15 it's installed by default.
- Check with `sysconfig.get_config_var("Py_GIL_DISABLED")` and `sys._is_gil_enabled()`.
- `concurrent.interpreters` and `InterpreterPoolExecutor` (3.14) run isolated interpreters in parallel.

## 🏋️ Practice
1. If you can install a free-threaded build (`python3.15t`), run `ft_bench.py` with both builds and compare.
2. Use `InterpreterPoolExecutor` to compute the sum of squares below 5,000,000 for 4 different offsets in parallel.